# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankita0531/ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb

import duckdb
import pandas as pd

con = duckdb.connect()

print("DuckDB connected.")

DuckDB connected.


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", HF_TOKEN is not None)

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Hugging Face authentication configured.")

HF token loaded: True
Hugging Face authentication configured.


In [3]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

print("March 2026 partition selected for baseline development.")

March 2026 partition selected for baseline development.


## 1. My rule and its reason codes

### Signal checks

**Signal 1 — GSC impressions (volume)**

**Verdict: CONFIRMED**

The volume buckets show substantial variation in search impressions, from low-volume content to 638,608 high-volume rows. Volume is also linked to the FlyRank quick-win logic from the session, so it is a reasonable signal for prioritization.

**Signal 2 — GSC average position**

**Verdict: CONFIRMED**

Among rows with GSC data available, the position buckets show clear separation across ranking strength: 2,183,484 rows are at positions 1–10, while 603,109 are at positions 31–100. This makes position a useful transparent signal for prioritizing content for review.

### Rule idea

I will prioritize content that has meaningful search visibility and weaker average search position. The rule uses a simple hand-written score rather than fitted model weights.

**Reason code:** `visible_low_position`

**Action:** `review_for_refresh`

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: GSC impressions (volume)

volume_buckets = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_data_available IS NOT TRUE THEN 'unavailable'
            WHEN gsc_impressions = 0 THEN 'zero'
            WHEN gsc_impressions < 10 THEN 'low (<10)'
            WHEN gsc_impressions < 100 THEN 'medium (10-99)'
            ELSE 'high (100+)'
        END AS impression_bucket,
        COUNT(*) AS n
    FROM {REL}
    GROUP BY 1
    ORDER BY
        CASE impression_bucket
            WHEN 'unavailable' THEN 1
            WHEN 'zero' THEN 2
            WHEN 'low (<10)' THEN 3
            WHEN 'medium (10-99)' THEN 4
            WHEN 'high (100+)' THEN 5
        END
""").df()

volume_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n
0,unavailable,6230317
1,low (<10),1463532
2,medium (10-99),1508921
3,high (100+),638608


In [5]:
# Signal 2: GSC average position

position_buckets = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_data_available IS NOT TRUE THEN 'unavailable'
            WHEN gsc_avg_position <= 10 THEN 'good (<=10)'
            WHEN gsc_avg_position <= 30 THEN 'middle (11-30)'
            WHEN gsc_avg_position <= 100 THEN 'weak (31-100)'
            ELSE 'very weak (100+)'
        END AS position_bucket,
        COUNT(*) AS n
    FROM {REL}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
    ORDER BY
        CASE position_bucket
            WHEN 'good (<=10)' THEN 1
            WHEN 'middle (11-30)' THEN 2
            WHEN 'weak (31-100)' THEN 3
            WHEN 'very weak (100+)' THEN 4
        END
""").df()

position_buckets

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n
0,good (<=10),2183484
1,middle (11-30),822477
2,weak (31-100),603109
3,very weak (100+),1991


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Build the ranked baseline queue

# Section 2: Build the ranked queue at client × content level

baseline_queue = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position
    FROM {REL}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
),

scored AS (
    SELECT
        *,
        CASE
            WHEN gsc_impressions >= 100
                 AND gsc_avg_position > 10
            THEN 2
            WHEN gsc_impressions >= 10
                 AND gsc_avg_position > 10
            THEN 1
            ELSE 0
        END AS score
    FROM monthly
)

SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    score,
    CASE
        WHEN score > 0 THEN 'visible_low_position'
        ELSE 'no_priority_signal'
    END AS reason_code,
    CASE
        WHEN score > 0 THEN 'review_for_refresh'
        ELSE 'monitor'
    END AS action
FROM scored
ORDER BY
    score DESC,
    gsc_impressions DESC
""").df()

print("Rows in ranked queue:", len(baseline_queue))
baseline_queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in ranked queue: 176738


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.173490,2,visible_low_position,review_for_refresh
1,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.786981,2,visible_low_position,review_for_refresh
2,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,60.0,22.143280,2,visible_low_position,review_for_refresh
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,197.0,23.591997,2,visible_low_position,review_for_refresh
4,client_23a62021009f63c4,content_66288edeb93b7c4f,137878.0,782.0,13.584096,2,visible_low_position,review_for_refresh
5,client_23a62021009f63c4,content_df47d1b976106de4,131707.0,163.0,24.426887,2,visible_low_position,review_for_refresh
6,client_23a62021009f63c4,content_5e1c049f62e33b11,120175.0,168.0,17.773364,2,visible_low_position,review_for_refresh
7,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,12.0,30.781284,2,visible_low_position,review_for_refresh
8,client_23a62021009f63c4,content_661a7734f691bef5,110424.0,73.0,24.816725,2,visible_low_position,review_for_refresh
9,client_23a62021009f63c4,content_cae701a83cad5e36,98572.0,242.0,23.609473,2,visible_low_position,review_for_refresh


In [10]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written to work/outputs/baseline_action_score.csv")

CSV written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The baseline ranks content using current-month search visibility and average search position. The confidence notes describe how strongly the available signals support the action; they are not model probabilities.

| Rank | Content | Action | Reason code | Confidence note | What would make it wrong |
|---|---|---|---|---|---|
| 1 | content_e8a52cf3d5988c07 | review_for_refresh | visible_low_position | High visibility with 244,931 impressions and position 15.17. | The average position may hide query-level variation or the page may already satisfy important queries. |
| 2 | content_36e53e9c707674fc | review_for_refresh | visible_low_position | High visibility with 194,579 impressions and weak position 32.79. | The impressions may come from low-value queries where a refresh would not help. |
| 3 | content_82e35c4845e6c391 | review_for_refresh | visible_low_position | 143,907 impressions with position 22.14 provide a clear visibility-and-ranking signal. | The content may not match the search intent behind its impressions. |
| 4 | content_3df3f32f3fd58dea | review_for_refresh | visible_low_position | 140,156 impressions and position 23.59 indicate substantial visibility with ranking room. | The average position may be influenced by a small set of queries. |
| 5 | content_66288edeb93b7c4f | review_for_refresh | visible_low_position | 137,878 impressions and position 13.58 indicate strong visibility just outside the strongest rankings. | The page may already perform well for its most valuable queries. |
| 6 | content_df47d1b976106de4 | review_for_refresh | visible_low_position | 131,707 impressions with position 24.43 indicate meaningful visibility and ranking opportunity. | A refresh may not improve performance if the underlying search intent has changed. |
| 7 | content_5e1c049f62e33b11 | review_for_refresh | visible_low_position | 120,175 impressions and position 17.77 support the refresh-priority rule. | The average position may not represent the highest-value queries. |
| 8 | content_bdf60c86117079be | review_for_refresh | visible_low_position | 112,429 impressions and position 30.78 indicate high visibility with weak ranking. | Only 12 clicks were recorded, so the impressions may have limited practical value. |
| 9 | content_661a7734f691bef5 | review_for_refresh | visible_low_position | 110,424 impressions and position 24.82 indicate substantial visibility with ranking room. | The impressions may not correspond to relevant or valuable searches. |
| 10 | content_cae701a83cad5e36 | review_for_refresh | visible_low_position | 98,572 impressions and position 23.61 support the visibility-plus-position rule. | Current traffic may already be satisfactory despite the average position. |
| 11 | content_559cdd76da9306de | review_for_refresh | visible_low_position | 97,378 impressions and position 36.90 indicate strong visibility but weak ranking. | Only 2 clicks suggest the impressions may have low value. |
| 12 | content_9fff53e827550f9d | review_for_refresh | visible_low_position | 94,673 impressions and position 23.50 provide meaningful visibility with ranking opportunity. | The content may be ranking for queries unrelated to the intended topic. |
| 13 | content_ba462518dad435fc | review_for_refresh | visible_low_position | 91,391 impressions and position 27.10 indicate substantial visibility and weaker ranking. | A refresh may not address the reason for the low ranking. |
| 14 | content_84a6bf3578312e90 | review_for_refresh | visible_low_position | 91,388 impressions and position 20.48 indicate meaningful visibility with improvement room. | The page may already be strong for its most important queries. |
| 15 | content_164c1f53f13bcee1 | review_for_refresh | visible_low_position | 89,982 impressions and position 23.32 support the baseline rule. | Only 2 clicks suggest that high impressions may not translate into useful traffic. |
| 16 | content_b51957d7f4abe47e | review_for_refresh | visible_low_position | 85,219 impressions and position 29.04 indicate visibility with weaker ranking. | The queries generating impressions may not be worth optimizing for. |
| 17 | content_6486239516a186d7 | review_for_refresh | visible_low_position | 83,293 impressions and position 29.83 indicate a clear ranking opportunity under the rule. | The content may have limited strategic value despite its impressions. |
| 18 | content_89c10d52fc81ac39 | review_for_refresh | visible_low_position | 81,777 impressions and position 23.72 indicate meaningful visibility with ranking room. | The average position may hide differences between valuable and low-value queries. |
| 19 | content_9d1e94cbd32b0a41 | review_for_refresh | visible_low_position | 81,479 impressions and position 34.53 indicate high visibility but weak ranking. | Only 13 clicks suggest the visibility may not represent valuable demand. |
| 20 | content_73aa61dcedebbf30 | review_for_refresh | visible_low_position | 80,124 impressions and position 46.38 indicate strong visibility with very weak ranking. | Only 9 clicks suggest that refreshing this content may not produce meaningful traffic. |

### Weak-pick observation

The review shows that some high-ranked candidates have very high impressions but extremely few clicks. For example, `content_bdf60c86117079be` has 112,429 impressions but only 12 clicks, while `content_559cdd76da9306de` has 97,378 impressions but only 2 clicks. These are potential weak picks because the baseline rule does not directly account for click-through rate or search intent.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Top-20 review candidates

top20 = baseline_queue.head(20).copy()

top20

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,score,reason_code,action
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931.0,669.0,15.173490,2,visible_low_position,review_for_refresh
1,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,242.0,32.786981,2,visible_low_position,review_for_refresh
2,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,60.0,22.143280,2,visible_low_position,review_for_refresh
3,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,197.0,23.591997,2,visible_low_position,review_for_refresh
4,client_23a62021009f63c4,content_66288edeb93b7c4f,137878.0,782.0,13.584096,2,visible_low_position,review_for_refresh
5,client_23a62021009f63c4,content_df47d1b976106de4,131707.0,163.0,24.426887,2,visible_low_position,review_for_refresh
6,client_23a62021009f63c4,content_5e1c049f62e33b11,120175.0,168.0,17.773364,2,visible_low_position,review_for_refresh
7,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,12.0,30.781284,2,visible_low_position,review_for_refresh
8,client_23a62021009f63c4,content_661a7734f691bef5,110424.0,73.0,24.816725,2,visible_low_position,review_for_refresh
9,client_23a62021009f63c4,content_cae701a83cad5e36,98572.0,242.0,23.609473,2,visible_low_position,review_for_refresh


## 4. Weak picks + leakage check

### Weak picks

Two candidates look weaker than the rest despite receiving high baseline scores:

- `content_bdf60c86117079be` — 112,429 impressions but only 12 clicks. The high visibility does not translate into much traffic, so a refresh may have limited value.
- `content_559cdd76da9306de` — 97,378 impressions but only 2 clicks and an average position of 36.90. This is a particularly weak pick because the baseline score does not consider click-through rate or search intent.

These examples show a limitation of the simple rule: high impressions plus a weak position does not always mean that refreshing the content will be useful.

### Leakage check

- **No future-window inputs:** The baseline queue uses only March 2026 GSC observations. No later-month performance or outcome was used to calculate the score.
- **No label-derived inputs:** The score is based only on `gsc_impressions` and `gsc_avg_position`. No target label or future outcome was included.
- **No product flags in the scoring rule:** The ranked score does not use product flags or post-outcome information.
- **Reason code and action are deterministic outputs of the same baseline rule.**

Therefore, the baseline ranking is based only on information that would be available at the decision moment.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Leakage verification

# Confirm the source window used by the baseline
date_check = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(*) AS rows_checked
    FROM {REL}
""").df()

print("Baseline source window:")
print(date_check)

# Confirm the scoring queue only contains current-signal fields
allowed_score_inputs = {
    "gsc_impressions",
    "gsc_avg_position"
}

queue_columns = set(baseline_queue.columns)

print("\nScore inputs:", allowed_score_inputs)
print("Queue columns:", sorted(queue_columns))

# No label/future/product-flag columns are used in the queue
forbidden_terms = ["label", "target", "future", "flag"]

for term in forbidden_terms:
    matches = [c for c in queue_columns if term in c.lower()]
    assert not matches, f"Potential leakage column found: {matches}"

# Confirm the baseline uses only March 2026 data
assert str(date_check.loc[0, "min_date"])[:7] == "2026-03"
assert str(date_check.loc[0, "max_date"])[:7] == "2026-03"

print("\nLeakage check passed: current March 2026 signals only.")
print("No label-derived, future-window, or product-flag columns are present in the scoring queue.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline source window:
    min_date   max_date  rows_checked
0 2026-03-01 2026-03-31       9841378

Score inputs: {'gsc_impressions', 'gsc_avg_position'}
Queue columns: ['action', 'client_hash_id', 'content_hash_id', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions', 'reason_code', 'score']

Leakage check passed: current March 2026 signals only.
No label-derived, future-window, or product-flag columns are present in the scoring queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.